# Análisis de los datos de Ausgrid

En este cuaderno se revisan los tres archivos anuales de Ausgrid *Solar Home Electricity Data*, con el consumo y la generación fotovoltaica de 300 hogares de Sídney entre julio de 2010 y junio de 2013. Se describe cómo están organizados, qué columnas tienen y qué valores contienen, y se comprueba si hay duplicados, fechas ausentes, celdas vacías o valores fuera de lo esperado. Al final se recogen los cambios necesarios para obtener un único conjunto con los tres años unido a ERA5.

Los archivos originales solo se leen. Las transformaciones se harán en los scripts de `src/preprocessing`.

## 1. Archivos y control de versión

Las huellas SHA-256 de los tres archivos están registradas en `docs/data/MANIFEST.md`. Se comprueban antes del análisis para asegurar que se trabaja con la misma versión que el resto del equipo. La primera línea de cada CSV es un título y no forma parte de la tabla, por lo que se omite al leer.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
raw_dir = project_root / "data" / "raw"

# Archivo, formato de la fecha y huella registrada en docs/data/MANIFEST.md
ausgrid_files = {
    "2010-2011": ("ausgrid_2010_2011.csv", "%d-%b-%y",
                  "2fc1f615e22fbd8ec350cdb19f7465d3a038453fc2ffd19473ef1bd38fd28289"),
    "2011-2012": ("ausgrid_2011_2012.csv", "%d/%m/%Y",
                  "a7d6abd1369311a5566e00d249141fd8aec28e9faba0bc928788607d961fc8c8"),
    "2012-2013": ("ausgrid_2012_2013.csv", "%d/%m/%Y",
                  "e4be46b3c9991c65735a9545319a2f1c33be8e639c75a17a88e0323a8e5098be"),
}

raw = {}
file_rows = []
for period, (file_name, date_format, expected_sha256) in ausgrid_files.items():
    path = raw_dir / file_name
    if not path.is_file():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")
    data = pd.read_csv(path, skiprows=1, low_memory=False)
    raw[period] = data
    file_rows.append({
        "período": period,
        "archivo": file_name,
        "huella coincide": hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha256,
        "filas": len(data),
        "columnas": data.shape[1],
    })
pd.DataFrame(file_rows).set_index("período")

,archivo,huella coincide,filas,columnas
período,,,,
2010-2011,ausgrid_2010_2011.csv,True,269735,53
2011-2012,ausgrid_2011_2012.csv,True,270304,54
2012-2013,ausgrid_2012_2013.csv,True,268557,54


**Resultado.** Las tres huellas coinciden con el manifiesto. Cada archivo tiene unas 270.000 filas. El de 2010-2011 tiene 53 columnas y los otros dos 54, porque incluyen la columna `Row Quality`.

## 2. Organización y columnas

Los archivos están en formato ancho: cada fila corresponde a un cliente, un día y una categoría de energía, y los valores del día se reparten en 48 columnas, una por cada media hora.

In [2]:
interval_columns = [f"{minutes // 60}:{minutes % 60:02d}" for minutes in range(30, 24 * 60, 30)] + ["0:00"]
example = raw["2010-2011"]
print("Columnas de 2010-2011:", example.columns[:5].to_list(), "+ 48 columnas de media hora:",
      interval_columns[0], "...", interval_columns[-1])
display(example.head(3))

column_info = pd.DataFrame(
    [
        ("Customer", "Identificador del cliente (1 a 300)", "—", str(example["Customer"].dtype)),
        ("Generator Capacity", "Potencia fotovoltaica instalada", "kWp", str(example["Generator Capacity"].dtype)),
        ("Postcode", "Código postal del hogar", "—", str(example["Postcode"].dtype)),
        ("Consumption Category", "Categoría de energía: GC, CL o GG", "—", "texto"),
        ("date", "Día del registro", "—", "texto"),
        ("0:30 ... 0:00", "Energía de cada media hora, etiquetada con el final del intervalo", "kWh por 30 min",
         str(example[interval_columns].dtypes.unique()[0])),
        ("Row Quality", "Calidad de la fila (solo 2011-2012 y 2012-2013)", "—", "vacía"),
    ],
    columns=["columna", "descripción", "unidad", "tipo de dato"],
).set_index("columna")
display(column_info)

same_columns = {
    period: data.columns.drop("Row Quality", errors="ignore").to_list() == example.columns.to_list()
    for period, data in raw.items()
}
print("Mismas columnas que 2010-2011, sin contar Row Quality:", same_columns)

Columnas de 2010-2011: ['Customer', 'Generator Capacity', 'Postcode', 'Consumption Category', 'date'] + 48 columnas de media hora: 0:30 ... 0:00


,Customer,Generator Capacity,Postcode,Consumption Category,date,0:30,1:00,1:30,2:00,2:30,...,19:30,20:00,20:30,21:00,21:30,22:00,22:30,23:00,23:30,0:00
0,1,3.78,2076,GC,1-Jul-10,0.303,0.471,0.083,0.121,0.361,...,0.495,0.54,0.406,0.543,0.495,0.216,0.378,0.128,0.078,0.125
1,1,3.78,2076,CL,1-Jul-10,1.250,1.244,1.256,0.744,0.019,...,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.075
2,1,3.78,2076,GG,1-Jul-10,0.000,0.000,0.000,0.000,0.000,...,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


,descripción,unidad,tipo de dato
columna,,,
Customer,Identificador del cliente (1 a 300),—,int64
Generator Capacity,Potencia fotovoltaica instalada,kWp,float64
Postcode,Código postal del hogar,—,int64
Consumption Category,"Categoría de energía: GC, CL o GG",—,texto
date,Día del registro,—,texto
0:30 ... 0:00,"Energía de cada media hora, etiquetada con el ...",kWh por 30 min,float64
Row Quality,Calidad de la fila (solo 2011-2012 y 2012-2013),—,vacía


Mismas columnas que 2010-2011, sin contar Row Quality: {'2010-2011': True, '2011-2012': True, '2012-2013': True}


**Resultado.** Los tres archivos tienen las mismas columnas, salvo `Row Quality`. Cada columna de media hora guarda la energía en kWh del intervalo que termina en la hora indicada: `0:30` es el intervalo de 00:00 a 00:30 y `0:00` es el de 23:30 a 00:00, que termina a la medianoche del día siguiente. Las columnas de energía son decimales; `date` y `Consumption Category` son texto. Para unir con ERA5 hay que pasar este formato ancho a una fila por intervalo, con su fecha y hora completas.

## 3. Categorías de energía

| Código | Nombre | Qué representa | Tratamiento |
|---|---|---|---|
| `GC` | *General Consumption* (consumo general) | Consumo eléctrico general del hogar | Forma parte de la demanda |
| `CL` | *Controlled Load* (carga controlada) | Consumo de una carga separada que funciona en horario valle, normalmente el calentador de agua. No todos los hogares la tienen | También es demanda: el consumo total del hogar es `GC + CL` |
| `GG` | *Gross Generation* (generación bruta) | Producción fotovoltaica total, medida por separado; no es el excedente exportado a la red | Es generación: se mantiene separada de la demanda |

Se cuentan las filas, los clientes y los días de cada categoría en cada año.

In [3]:
coverage_rows = []
for period, data in raw.items():
    for category, group in data.groupby("Consumption Category"):
        coverage_rows.append({
            "período": period,
            "categoría": category,
            "filas": len(group),
            "clientes": group["Customer"].nunique(),
            "días": group["date"].nunique(),
        })
coverage = pd.DataFrame(coverage_rows).set_index(["período", "categoría"])
coverage

filas  clientes  días
período   categoría                        
2010-2011 CL          50735       139   365
          GC         109500       300   365
          GG         109500       300   365
2011-2012 CL          50704       139   366
          GC         109800       300   366
          GG         109800       300   366
2012-2013 CL          49719       138   365
          GC         109419       300   365
          GG         109419       300   365

**Resultado.** `GC` y `GG` están presentes para los 300 clientes en los tres años. `CL` aparece en 139 clientes en los dos primeros años y en 138 en el tercero: el cliente 27 deja de tener carga controlada. La ausencia de `CL` en un cliente significa que ese hogar no tiene esa carga, no que falten datos. Si un cliente tiene `CL`, hay que comprobar además que tenga la fila todos los días (sección 8).

## 4. Datos fijos de cada cliente

La capacidad instalada y el código postal describen el hogar y deben ser iguales en todas sus filas y en los tres años.

In [4]:
customer_info = pd.concat(
    [data.groupby("Customer")[["Generator Capacity", "Postcode"]].agg(["nunique", "first"]).assign(período=period)
     for period, data in raw.items()]
)
values_per_customer = customer_info.xs("nunique", axis=1, level=1).drop(columns="período", errors="ignore")
first_values = customer_info.xs("first", axis=1, level=1)
changes_between_years = first_values.groupby(level=0).nunique()

print("Clientes con más de un valor dentro de un mismo año:", int((values_per_customer > 1).any(axis=1).sum()))
print("Clientes con valores distintos entre años:", int((changes_between_years > 1).any(axis=1).sum()))
print("Capacidad instalada (kWp): mínimo", first_values["Generator Capacity"].min(),
      "| mediana", first_values["Generator Capacity"].median(),
      "| máximo", first_values["Generator Capacity"].max())
print("Códigos postales distintos:", first_values["Postcode"].nunique())

Clientes con más de un valor dentro de un mismo año: 0
Clientes con valores distintos entre años: 0
Capacidad instalada (kWp): mínimo 1.0 | mediana 1.5 | máximo 9.99
Códigos postales distintos: 100


**Resultado.** Cada cliente tiene una sola capacidad y un solo código postal, sin cambios entre años. La capacidad va de 1,0 a 9,99 kWp. Estos datos se pueden guardar una sola vez por cliente.

## 5. Fechas

Los archivos no usan el mismo formato de fecha: 2010-2011 usa `1-Jul-10` (`%d-%b-%y`) y los otros dos `1/07/2011` (`%d/%m/%Y`). En este cuaderno la fecha se deja como texto; la conversión a fecha se hará en el script de preprocesamiento, con el formato de cada archivo. Aquí se comprueba cuántas fechas distintas tiene cada archivo, cuáles son la primera y la última y si las filas aparecen en orden.

Para poder comparar días sin convertir el texto, a cada fecha se le asigna un número de día según el orden en que aparece en el archivo: 0 para el 1 de julio, 1 para el 2 de julio, y así sucesivamente.

In [5]:
date_rows = []
file_dates = {}
for period, data in raw.items():
    dates = list(data["date"].unique())  # en el orden en que aparecen en el archivo
    file_dates[period] = dates
    data["day_number"] = data["date"].map({date: number for number, date in enumerate(dates)})
    rows_in_order = data.groupby(["Customer", "Consumption Category"])["day_number"].apply(
        lambda numbers: numbers.is_monotonic_increasing
    ).all()
    date_rows.append({
        "período": period,
        "formato": ausgrid_files[period][1],
        "tipo de dato": "texto" if data["date"].dtype == object else str(data["date"].dtype),
        "primera fecha": dates[0],
        "última fecha": dates[-1],
        "fechas distintas": len(dates),
        "filas en orden de fecha": bool(rows_in_order),
    })
pd.DataFrame(date_rows).set_index("período")

,formato,tipo de dato,primera fecha,última fecha,fechas distintas,filas en orden de fecha
período,,,,,,
2010-2011,%d-%b-%y,texto,1-Jul-10,30-Jun-11,365,True
2011-2012,%d/%m/%Y,texto,1/07/2011,30/06/2012,366,True
2012-2013,%d/%m/%Y,texto,1/07/2012,30/06/2013,365,True


**Resultado.** La fecha está guardada como texto en los tres archivos. Cada archivo va del 1 de julio al 30 de junio y tiene tantas fechas distintas como días tiene ese año: 365, 366 y 365. Dentro de cada cliente y categoría las filas aparecen en orden de fecha, así que el número de día sirve para ordenar y comparar días en las secciones siguientes.

La conversión a fecha se hará en la función `parse_dates` del script de preprocesamiento, con `%d-%b-%y` para 2010-2011 y `%d/%m/%Y` para los otros dos años. Un único formato para los tres archivos no funcionaría. Después de convertirlas se comprobará que ninguna fecha quede sin interpretar y que las tres tablas usen el mismo tipo de dato.

## 6. Registros duplicados

La clave de cada fila es cliente + fecha + categoría. Se buscan claves repetidas en cada archivo.

In [6]:
duplicate_rows = []
for period, data in raw.items():
    duplicated = data.duplicated(subset=["Customer", "date", "Consumption Category"], keep=False)
    duplicate_rows.append({"período": period, "filas con clave repetida": int(duplicated.sum())})
pd.DataFrame(duplicate_rows).set_index("período")

,filas con clave repetida
período,
2010-2011,0
2011-2012,0
2012-2013,0


**Resultado.** No hay claves repetidas en ningún año, por lo que no hace falta eliminar duplicados. La unicidad se volverá a comprobar después de construir las marcas de tiempo, con la clave cliente + marca UTC.

## 7. Días de cliente ausentes

Cada cliente debería tener una fila de `GC` y otra de `GG` para cada día del año. Se compara lo observado con el calendario completo.

In [7]:
calendar_rows = []
missing_blocks = []
for period, data in raw.items():
    dates = file_dates[period]
    expected = pd.MultiIndex.from_product([sorted(data["Customer"].unique()), range(len(dates))])
    for category in ("GC", "GG"):
        rows = data.loc[data["Consumption Category"].eq(category)]
        missing = expected.difference(pd.MultiIndex.from_frame(rows[["Customer", "day_number"]]))
        calendar_rows.append({
            "período": period,
            "categoría": category,
            "días esperados": len(expected),
            "días ausentes": len(missing),
            "porcentaje": round(100 * len(missing) / len(expected), 3),
        })
        if category == "GC" and len(missing):
            missing_frame = missing.to_frame(index=False, name=["cliente", "día"])
            for customer, group in missing_frame.groupby("cliente"):
                first, last = group["día"].min(), group["día"].max()
                missing_blocks.append({
                    "período": period,
                    "cliente": customer,
                    "días": len(group),
                    "desde": dates[first],
                    "hasta": dates[last],
                    "bloque continuo": len(group) == last - first + 1,
                })
display(pd.DataFrame(calendar_rows).set_index(["período", "categoría"]))
display(pd.DataFrame(missing_blocks))

días esperados  días ausentes  porcentaje
período   categoría                                           
2010-2011 GC                 109500              0       0.000
          GG                 109500              0       0.000
2011-2012 GC                 109800              0       0.000
          GG                 109800              0       0.000
2012-2013 GC                 109500             81       0.074
          GG                 109500             81       0.074

,período,cliente,días,desde,hasta,bloque continuo
0,2012-2013,2,81,12/10/2012,31/12/2012,True


**Resultado.** En 2010-2011 y 2011-2012 no falta ningún día. En 2012-2013 faltan 81 días del cliente 2, del 12 de octubre al 31 de diciembre de 2012, en un solo bloque continuo. En esos días faltan a la vez `GC`, `GG` y `CL`, y representan el 0,074 % de los días de ese año. Esos días quedarán ausentes en el conjunto final: no se rellenarán con ceros ni se interpolarán, y el resto del historial del cliente 2 se conserva.

## 8. Filas de `CL` incompletas

Para calcular el consumo total como `GC + CL` hace falta que cada cliente con carga controlada tenga la fila `CL` todos los días en que tiene `GC`.

In [8]:
cl_rows = []
for period, data in raw.items():
    gc_keys = data.loc[data["Consumption Category"].eq("GC"), ["Customer", "date"]]
    cl_keys = data.loc[data["Consumption Category"].eq("CL"), ["Customer", "date"]]
    gc_of_cl_customers = gc_keys.loc[gc_keys["Customer"].isin(cl_keys["Customer"].unique())]
    missing = pd.MultiIndex.from_frame(gc_of_cl_customers).difference(pd.MultiIndex.from_frame(cl_keys))
    missing_customers = sorted(set(missing.get_level_values(0)))
    cl_rows.append({
        "período": period,
        "clientes con CL": cl_keys["Customer"].nunique(),
        "días con GC pero sin CL": len(missing),
        "clientes afectados": len(missing_customers),
        "clientes": [int(c) for c in missing_customers],
    })
pd.DataFrame(cl_rows).set_index("período")

,clientes con CL,días con GC pero sin CL,clientes afectados,clientes
período,,,,
2010-2011,139,0,0,[]
2011-2012,139,170,1,[27]
2012-2013,138,570,10,"[68, 95, 161, 187, 248, 272, 284, 289, 293, 294]"


**Resultado.** En 2010-2011 todos los clientes con carga controlada tienen la fila `CL` todos los días. En 2011-2012 faltan 170 días del cliente 27 y en 2012-2013 faltan 570 días repartidos entre 10 clientes. En total son 740 días de cliente, el 0,2 % de los días con `GC`.

Antes de decidir qué hacer con esos días se revisa cómo están repartidos y cuánta carga controlada usa cada cliente afectado.

In [9]:
detail_rows = []
for period, data in raw.items():
    dates = file_dates[period]
    gc = data.loc[data["Consumption Category"].eq("GC")]
    cl = data.loc[data["Consumption Category"].eq("CL")]
    for customer in sorted(cl["Customer"].unique()):
        cl_customer = cl.loc[cl["Customer"].eq(customer)]
        missing = sorted(set(gc.loc[gc["Customer"].eq(customer), "day_number"]) - set(cl_customer["day_number"]))
        if not missing:
            continue
        blocks = 1 + sum(1 for a, b in zip(missing, missing[1:]) if b != a + 1)
        daily_cl = cl_customer[interval_columns].sum(axis=1)
        detail_rows.append({
            "período": period,
            "cliente": int(customer),
            "días sin CL": len(missing),
            "bloques": blocks,
            "desde": dates[missing[0]],
            "hasta": dates[missing[-1]],
            "CL medio registrado (kWh/día)": round(float(daily_cl.mean()), 2),
            "días registrados con CL = 0 (%)": round(100 * float((daily_cl == 0).mean()), 1),
        })
cl_detail = pd.DataFrame(detail_rows)
cl_detail

,período,cliente,días sin CL,bloques,desde,hasta,CL medio registrado (kWh/día),días registrados con CL = 0 (%)
0,2011-2012,27,170,1,13/01/2012,30/06/2012,0.00,100.0
1,2012-2013,68,65,1,26/01/2013,31/03/2013,1.58,25.3
2,2012-2013,95,47,1,13/02/2013,31/03/2013,11.46,0.0
3,2012-2013,161,59,1,1/02/2013,31/03/2013,3.47,37.6
4,2012-2013,187,58,1,2/02/2013,31/03/2013,4.10,6.5
5,2012-2013,248,45,1,15/02/2013,31/03/2013,4.71,0.0
6,2012-2013,272,65,1,26/01/2013,31/03/2013,1.55,34.7
7,2012-2013,284,65,1,26/01/2013,31/03/2013,0.25,85.0
8,2012-2013,289,60,1,31/01/2013,31/03/2013,8.56,0.0
9,2012-2013,293,48,1,12/02/2013,31/03/2013,7.43,0.0


El `CL` del cliente 27 vale 0 en todos los días registrados de 2011-2012, así que se buscan todos los clientes que tienen la fila `CL` pero con valor 0 todo el año. Para los clientes de 2012-2013 se comprueba si tienen `CL` en las mismas fechas del año anterior, que sería la base para estimar los días que faltan.

In [10]:
# Clientes que tienen la fila CL, pero con todos sus valores del año iguales a 0.
zero_cl_rows = []
for period, data in raw.items():
    cl = data.loc[data["Consumption Category"].eq("CL")]
    yearly_cl = cl.groupby("Customer")[interval_columns].sum().sum(axis=1)
    for customer in yearly_cl.index[yearly_cl.eq(0)]:
        zero_cl_rows.append({
            "período": period,
            "cliente": int(customer),
            "días con fila CL": int(cl["Customer"].eq(customer).sum()),
            "CL total del año (kWh)": 0.0,
        })
display(pd.DataFrame(zero_cl_rows))

previous = raw["2011-2012"]
previous_cl = previous.loc[previous["Consumption Category"].eq("CL")]
previous_rows = []
for _, row in cl_detail.loc[cl_detail["período"].eq("2012-2013")].iterrows():
    dates = file_dates["2012-2013"]
    missing_dates = dates[dates.index(row["desde"]): dates.index(row["hasta"]) + 1]
    same_dates_2012 = [date.replace("/2013", "/2012") for date in missing_dates]  # mismo día y mes, un año antes
    available = previous_cl.loc[previous_cl["Customer"].eq(row["cliente"]) & previous_cl["date"].isin(same_dates_2012)]
    previous_rows.append({
        "cliente": row["cliente"],
        "días sin CL en 2013": row["días sin CL"],
        "mismos días con CL en 2012": len(available),
        "CL medio en esos días de 2012 (kWh/día)": round(float(available[interval_columns].sum(axis=1).mean()), 2),
        "CL medio registrado en 2012-2013 (kWh/día)": row["CL medio registrado (kWh/día)"],
    })
pd.DataFrame(previous_rows).set_index("cliente")

,período,cliente,días con fila CL,CL total del año (kWh)
0,2010-2011,27,365,0.0
1,2010-2011,37,365,0.0
2,2010-2011,281,365,0.0
3,2011-2012,27,196,0.0
4,2011-2012,37,366,0.0
5,2011-2012,281,366,0.0
6,2012-2013,112,365,0.0
7,2012-2013,158,365,0.0
8,2012-2013,281,365,0.0


,días sin CL en 2013,mismos días con CL en 2012,CL medio en esos días de 2012 (kWh/día),CL medio registrado en 2012-2013 (kWh/día)
cliente,,,,
68,65,65,1.71,1.58
95,47,47,10.15,11.46
161,59,59,1.93,3.47
187,58,58,3.11,4.10
248,45,45,4.19,4.71
272,65,65,0.80,1.55
284,65,65,0.00,0.25
289,60,60,8.27,8.56
293,48,48,5.73,7.43


**Resultado.** Los días sin `CL` corresponden a dos casos distintos, y cada uno se trata de una forma.

**Caso 1: clientes con `CL` siempre igual a 0.** El cliente 27 tiene `CL` igual a 0 en todos los días registrados de 2010-2011 y 2011-2012. Desde el 13 de enero de 2012 la fila `CL` deja de aparecer y en 2012-2013 ya no existe. No es el único caso: hay clientes que tienen la fila `CL` todos los días pero siempre con 0. Son el 37 y el 281 en 2010-2011 y 2011-2012, y el 112, el 158 y el 281 en 2012-2013. Son hogares con contador de carga controlada que no la usan. Como esa carga siempre vale cero, no hay nada que estimar: estos clientes se tratan como hogares sin carga controlada y su consumo total es `GC`. Por eso cuentan como clientes con carga controlada 136 de los 139 que tienen fila `CL` en los dos primeros años, y 135 de los 138 en 2012-2013.

**Caso 2: clientes 68, 95, 161, 187, 248, 272, 284, 289, 293 y 294.** Cada uno tiene un solo bloque de 45 a 65 días sin `CL`, y los diez bloques terminan el mismo día, el 31 de marzo de 2013. Que coincidan en la fecha final apunta a un fallo en el registro de `CL`, no a que los hogares apagaran la carga. Varios de estos clientes usan mucha carga controlada; el cliente 95, por ejemplo, registra 11,5 kWh al día, así que usar solo `GC` en esos días subestimaría su demanda. Por eso el `CL` de estos días se estimará: cada media hora ausente se rellenará con el `CL` del mismo cliente en la misma fecha y la misma media hora del año anterior. Los diez clientes tienen esas fechas completas en 2012, y sus valores son del mismo orden que los registrados en 2012-2013. Así se mantienen los hábitos de cada hogar, la época del año y la forma del día. Los días estimados llevarán la marca `cl_estimated`: se usarán para entrenar los modelos, pero no para evaluarlos, porque el error se mediría contra un valor que no es real.

El consumo total se calculará con estas reglas:

| Caso | Consumo total | Marca |
|---|---|---|
| Fila `CL` presente | `GC + CL` | — |
| Cliente sin carga controlada en el año, o con `CL` siempre igual a 0 (27, 37 y 281 en los dos primeros años; 112, 158 y 281 en 2012-2013) | `GC` | `has_cl = False` |
| Día sin `CL` de los 10 clientes de 2012-2013 | `GC + CL` del mismo día del año anterior | `cl_estimated = True` |
| Día sin `GC` (cliente 2, sección 7) | Vacío: falta todo el registro | Fila ausente |

Con estas reglas ningún día queda sin consumo total por falta de `CL`.

## 9. Celdas vacías, negativos y ceros

Se cuentan las celdas vacías, los valores negativos y los ceros en las 48 columnas de energía, por año y categoría. También se revisa el contenido de `Row Quality`.

In [11]:
quality_rows = []
for period, data in raw.items():
    for category, group in data.groupby("Consumption Category"):
        values = group[interval_columns].to_numpy(dtype=float)
        quality_rows.append({
            "período": period,
            "categoría": category,
            "celdas": values.size,
            "vacías": int(np.isnan(values).sum()),
            "negativas": int((values < 0).sum()),
            "ceros (%)": round(100 * (values == 0).mean(), 2),
        })
display(pd.DataFrame(quality_rows).set_index(["período", "categoría"]))

for period, data in raw.items():
    if "Row Quality" in data.columns:
        print(f"{period}: Row Quality vacía en {int(data['Row Quality'].isna().sum())} de {len(data)} filas")

celdas  vacías  negativas  ceros (%)
período   categoría                                       
2010-2011 CL         2435280       0          0      89.60
          GC         5256000       0          0       0.21
          GG         5256000       0          0      53.67
2011-2012 CL         2433792       0          0      89.51
          GC         5270400       0          0       0.12
          GG         5270400       0          0      54.29
2012-2013 CL         2386512       0          0      89.80
          GC         5252112       0          0       0.33
          GG         5252112       0          0      54.43

2011-2012: Row Quality vacía en 270304 de 270304 filas
2012-2013: Row Quality vacía en 268557 de 268557 filas


**Resultado.** No hay celdas vacías ni valores negativos. Los ceros son frecuentes en `GG`, alrededor del 54 %, porque de noche no hay generación, y en `CL`, alrededor del 90 %, porque la carga controlada solo funciona unas horas al día. En `GC` los ceros no llegan al 0,4 %. Todos son valores medidos y se conservan. `Row Quality` está vacía en todas las filas, así que no aporta información y se eliminará.

## 10. Rangos de valores

Se resumen el mínimo, la mediana, el percentil 99 y el máximo de la energía por media hora, y se localiza el máximo de cada categoría y año.

In [12]:
range_rows = []
for period, data in raw.items():
    for category, group in data.groupby("Consumption Category"):
        values = group[interval_columns].to_numpy(dtype=float)
        row, column = np.unravel_index(values.argmax(), values.shape)
        record = group.iloc[row]
        p99 = float(np.quantile(values, 0.99))
        range_rows.append({
            "período": period,
            "categoría": category,
            "mínimo": values.min(),
            "mediana": float(np.median(values)),
            "p99": round(p99, 3),
            "máximo": values.max(),
            "máximo / p99": round(values.max() / p99, 2),
            "cliente": record["Customer"],
            "fecha": record["date"],
            "intervalo": interval_columns[column],
            "capacidad (kWp)": record["Generator Capacity"],
        })
pd.DataFrame(range_rows).set_index(["período", "categoría"])

mínimo  mediana    p99  máximo  máximo / p99  cliente  \
período   categoría                                                          
2010-2011 CL            0.0    0.000  2.352   5.059          2.15      247   
          GC            0.0    0.212  2.070   7.073          3.42      120   
          GG            0.0    0.000  1.031   4.300          4.17       75   
2011-2012 CL            0.0    0.000  2.315   5.201          2.25      300   
          GC            0.0    0.199  1.911   7.805          4.08       85   
          GG            0.0    0.000  1.019   4.419          4.34       75   
2012-2013 CL            0.0    0.000  2.215   4.091          1.85      300   
          GC            0.0    0.192  1.935   6.567          3.39        8   
          GG            0.0    0.000  1.038   4.331          4.17       75   

                          fecha intervalo  capacidad (kWp)  
período   categoría                                         
2010-2011 CL          27-Sep-10      2:00             1.02  
          GC           1-Dec-10     21:30             2.04  
          GG          20-Dec-10     14:00             8.00  
2011-2012 CL          1/04/2012      2:30             3.36  
          GC          9/01/2012     20:00             2.00  
          GG         14/02/2012     13:00             8.00  
2012-2013 CL          7/04/2013      2:30             3.36  
          GC          2/08/2012     18:30             1.02  
          GG         14/01/2013     13:30             8.00

**Resultado.** La mediana de `GC` es de unos 0,2 kWh por media hora, es decir, unos 0,4 kW de potencia media. Los máximos de `GC` llegan a 7,8 kWh por media hora, unos 15,6 kW, entre 3 y 4 veces el p99. Es un valor alto para un hogar pero posible con varios equipos grandes encendidos a la vez, y no hay otra información que permita descartarlo, así que se conserva. Los máximos de `CL` de 2011-2012 y 2012-2013 caen en el intervalo de las 02:30 del 1 de abril de 2012 y del 7 de abril de 2013, que son días de fin del horario de verano; la sección 13 explica por qué esos valores son dobles. Los máximos de `GG` se comparan con la capacidad instalada en la sección siguiente.

## 11. Generación frente a la capacidad instalada

Un sistema de `C` kWp puede generar como mucho unos `C × 0,5` kWh en media hora. Se cuentan las medias horas en que `GG` supera ese valor y los clientes en que lo supera en más de un 20 %.

In [13]:
capacity_rows = []
over_capacity_customers = {}
for period, data in raw.items():
    gg = data.loc[data["Consumption Category"].eq("GG")]
    ratio = gg[interval_columns].to_numpy(dtype=float) / (gg["Generator Capacity"].to_numpy()[:, None] * 0.5)
    over_20 = gg.loc[(ratio > 1.2).any(axis=1), "Customer"]
    over_capacity_customers[period] = sorted(int(c) for c in over_20.unique())
    capacity_rows.append({
        "período": period,
        "medias horas con GG > capacidad": int((ratio > 1).sum()),
        "porcentaje de medias horas": round(100 * (ratio > 1).mean(), 3),
        "medias horas con GG > 1,2 × capacidad": int((ratio > 1.2).sum()),
        "clientes por encima de 1,2": len(over_capacity_customers[period]),
        "relación máxima": round(float(ratio.max()), 2),
    })
display(pd.DataFrame(capacity_rows).set_index("período"))
print("Clientes por encima de 1,2 en algún año:", sorted(int(c) for c in set().union(*over_capacity_customers.values())))

,medias horas con GG > capacidad,porcentaje de medias horas,"medias horas con GG > 1,2 × capacidad","clientes por encima de 1,2",relación máxima
período,,,,,
2010-2011,3418,0.065,906,6,1.48
2011-2012,3806,0.072,1111,5,1.51
2012-2013,5354,0.102,1366,14,1.65


Clientes por encima de 1,2 en algún año: [23, 50, 60, 82, 112, 137, 148, 199, 215, 219, 221, 262, 265, 285, 299]


**Resultado.** Superar ligeramente la capacidad nominal es normal: los paneles pueden rendir algo más en días fríos y despejados. Lo que no es normal es superarla en más de un 20 %, y eso ocurre en 15 clientes, con relaciones de hasta 1,65. Lo más probable es que la capacidad registrada de esos clientes sea menor que la real. Los valores de `GG` no se modifican, pero esos clientes se marcarán para no usarlos al validar el modelo de generación con pvlib, que depende de la capacidad.

## 12. Generación nocturna

Entre las 00:00 y las 04:00 no hay sol en Sídney en ninguna época del año, por lo que `GG` debería ser cero. Se cuentan las medias horas de ese tramo con generación y se separan los valores muy pequeños de los que no lo son.

In [14]:
night_columns = interval_columns[:8]  # intervalos que terminan entre las 00:30 y las 04:00
night_rows = []
for period, data in raw.items():
    gg = data.loc[data["Consumption Category"].eq("GG")]
    night = gg[night_columns].to_numpy(dtype=float)
    positive = night[night > 0]
    large = gg.loc[(gg[night_columns] > 0.05).any(axis=1)]
    night_rows.append({
        "período": period,
        "medias horas nocturnas con GG > 0": len(positive),
        "mediana de esos valores (kWh)": float(np.median(positive)) if len(positive) else 0,
        "medias horas con GG > 0,05 kWh": int((night > 0.05).sum()),
        "clientes con GG > 0,05 kWh": sorted(int(c) for c in large["Customer"].unique()),
    })
display(pd.DataFrame(night_rows).set_index("período"))

# Días con generación desplazada: más del 20 % de la energía del día entre las 00:00 y las 05:00.
shifted_rows = []
for period, data in raw.items():
    gg = data.loc[data["Consumption Category"].eq("GG")]
    daily = gg[interval_columns].sum(axis=1)
    early = gg[interval_columns[:10]].sum(axis=1)
    shifted = gg.loc[(early > 0.2 * daily) & (daily > 0.5)]
    for customer, group in shifted.groupby("Customer"):
        shifted_rows.append({
            "período": period,
            "cliente": customer,
            "días": len(group),
            "desde": group["date"].iloc[0],
            "hasta": group["date"].iloc[-1],
        })
pd.DataFrame(shifted_rows)

,medias horas nocturnas con GG > 0,mediana de esos valores (kWh),"medias horas con GG > 0,05 kWh","clientes con GG > 0,05 kWh"
período,,,,
2010-2011,10937,0.006,0,[]
2011-2012,10198,0.006,62,[146]
2012-2013,9892,0.006,696,"[97, 146]"


,período,cliente,días,desde,hasta
0,2011-2012,146,16,13/06/2012,30/06/2012
1,2012-2013,97,68,13/07/2012,19/09/2012
2,2012-2013,146,47,1/07/2012,16/08/2012


**Resultado.** Hay dos situaciones distintas:

- **Valores muy pequeños.** Unas 10.000 medias horas nocturnas al año tienen `GG` entre 0,006 y 0,013 kWh. Son el escalón mínimo del contador y corresponden a ruido de medida o al consumo del inversor. Su efecto en la energía es despreciable y se conservan.
- **Generación desplazada.** Los clientes 97 y 146 tienen varias semanas entre junio y septiembre de 2012 en que la curva de generación aparece de madrugada, con valores de hasta 0,9 kWh por media hora. La forma de la curva es la de un día solar normal, pero desplazada varias horas, lo que apunta a un error en la hora del registro y no a una generación real. Son 131 días en total: 63 del cliente 146 (del 13 de junio al 16 de agosto de 2012) y 68 del cliente 97 (del 13 de julio al 19 de septiembre de 2012). Esos días de `GG` se marcarán para no usarlos al validar pvlib. Su `GC` se conserva.

## 13. Horario de verano

Para unir con ERA5 hay que pasar la hora de Ausgrid a UTC, y eso depende de si los contadores registran la hora local con horario de verano o la hora estándar fija. Se comprueba con el centro de la curva de generación, que debe quedar cerca del mediodía solar. Si los registros siguen el horario de verano, ese centro debe saltar una hora en los días de cambio.

In [15]:
dst_changes = [  # cambios de horario en Nueva Gales del Sur, con la fecha escrita como en cada archivo
    ("2010-2011", "3-Oct-10", "inicio"), ("2010-2011", "3-Apr-11", "fin"),
    ("2011-2012", "2/10/2011", "inicio"), ("2011-2012", "1/04/2012", "fin"),
    ("2012-2013", "7/10/2012", "inicio"), ("2012-2013", "7/04/2013", "fin"),
]
interval_middle = np.arange(1, 49) * 0.5 - 0.25  # centro de cada media hora, en horas

generation_center = {}
for period, data in raw.items():
    gg = data.loc[data["Consumption Category"].eq("GG") & ~data["Customer"].isin([97, 146])]
    profile = gg.groupby("day_number")[interval_columns].mean().to_numpy()
    generation_center[period] = pd.Series((profile * interval_middle).sum(axis=1) / profile.sum(axis=1))

dst_rows = []
for period, day, change in dst_changes:
    number = file_dates[period].index(day)
    center = generation_center[period]
    before = center.loc[number - 7: number - 1].mean()
    after = center.loc[number + 1: number + 7].mean()
    dst_rows.append({
        "día de cambio": day,
        "cambio": change,
        "centro de GG, semana anterior (h)": round(before, 2),
        "centro de GG, semana siguiente (h)": round(after, 2),
        "diferencia (h)": round(after - before, 2),
    })
pd.DataFrame(dst_rows).set_index("día de cambio")

,cambio,"centro de GG, semana anterior (h)","centro de GG, semana siguiente (h)",diferencia (h)
día de cambio,,,,
3-Oct-10,inicio,11.58,12.79,1.20
3-Apr-11,fin,12.98,11.95,-1.03
2/10/2011,inicio,11.65,12.64,0.99
1/04/2012,fin,12.96,12.06,-0.89
7/10/2012,inicio,11.81,13.00,1.19
7/04/2013,fin,13.07,11.89,-1.17


**Resultado.** El centro de la generación salta cerca de +1 h al empezar el horario de verano y −1 h al terminar. Los registros de Ausgrid están en hora local de Sídney con horario de verano. Para pasar a UTC se usará la zona `Australia/Sydney`, que aplica los cambios automáticamente.

Queda por ver qué ocurre en los propios días de cambio, en los que el reloj salta o repite una hora y, aun así, el archivo tiene 48 medias horas. Se comparan los intervalos de las 02:00 a las 03:00 con los de la semana anterior.

In [16]:
dst_slots = ["2:30", "3:00"]  # intervalos de 02:00 a 03:00
slot_rows = []
for period, day, change in dst_changes:
    data = raw[period]
    gc = data.loc[data["Consumption Category"].eq("GC")]
    number = file_dates[period].index(day)
    today = gc.loc[gc["day_number"].eq(number)]
    week_before = gc.loc[gc["day_number"].eq(number - 7)]
    slot_rows.append({
        "día de cambio": day,
        "cambio": change,
        "filas del día": len(today),
        "medias horas con dato": int(today[interval_columns].notna().sum(axis=1).min()),
        "GC medio 02:00-03:00 (kWh)": round(float(today[dst_slots].mean().mean()), 3),
        "GC medio una semana antes (kWh)": round(float(week_before[dst_slots].mean().mean()), 3),
        "filas con cero en 02:00-03:00 (%)": round(100 * float((today[dst_slots] == 0).all(axis=1).mean()), 1),
    })
pd.DataFrame(slot_rows).set_index("día de cambio")

,cambio,filas del día,medias horas con dato,GC medio 02:00-03:00 (kWh),GC medio una semana antes (kWh),filas con cero en 02:00-03:00 (%)
día de cambio,,,,,,
3-Oct-10,inicio,300,48,0.000,0.174,100.0
3-Apr-11,fin,300,48,0.353,0.176,0.0
2/10/2011,inicio,300,48,0.000,0.173,100.0
1/04/2012,fin,300,48,0.366,0.175,0.0
7/10/2012,inicio,300,48,0.000,0.185,100.0
7/04/2013,fin,300,48,0.366,0.190,0.0


**Resultado.** Todos los días de cambio tienen las 48 medias horas, pero su contenido depende del tipo de cambio:

- **Inicio del horario de verano.** El reloj pasa de las 02:00 a las 03:00, así que esa hora no existe. Las columnas `2:30` y `3:00` valen 0 en todas las filas. Esas dos medias horas se eliminarán al construir las marcas de tiempo; la energía del día no cambia, porque son ceros.
- **Fin del horario de verano.** El reloj vuelve de las 03:00 a las 02:00, así que esa hora ocurre dos veces. Las columnas `2:30` y `3:00` tienen aproximadamente el doble de energía que una semana antes, porque suman las dos pasadas. Cada uno de esos valores se repartirá a partes iguales entre las dos pasadas de la hora, que en UTC son horas distintas. Así la energía del día se conserva.

## 14. Lo que estos datos no contienen

Los hogares de Ausgrid están conectados a la red y no tienen batería. Los archivos no incluyen el estado de carga de una batería ni marcas de cortes de suministro, así que un cero en `GC` no indica un corte ni una batería agotada. La energía no servida se obtendrá más adelante con una simulación de la batería a partir de los pronósticos de demanda y generación; no se crea ninguna marca de corte durante el preprocesamiento.

## 15. Resumen de hallazgos

| Aspecto | Resultado | ¿Requiere cambio? |
|---|---|---|
| Versión de los archivos | Las tres huellas SHA-256 coinciden con el manifiesto | No |
| Organización | Formato ancho: una fila por cliente, día y categoría, con 48 columnas de media hora | Sí: pasar a una fila por intervalo |
| Primera línea del CSV | Es un título, no la cabecera | Sí: omitirla al leer |
| Fechas | Texto, con un formato en 2010-2011 (`%d-%b-%y`) y otro en los demás años (`%d/%m/%Y`) | Sí: convertir a fecha en `parse_dates` con el formato de cada archivo |
| Columna `0:00` | Intervalo que termina a la medianoche del día siguiente | Sí: asignarla al día siguiente |
| Horario de verano | Hora local con cambios; ceros en la hora que no existe y valores dobles en la que se repite | Sí: quitar los ceros, repartir los dobles y pasar a UTC |
| `Row Quality` | Vacía en todas las filas | Sí: eliminarla |
| Datos fijos del cliente | Capacidad y código postal constantes | No |
| Duplicados | 0 claves repetidas | No |
| Celdas vacías y negativos | 0 y 0 | No |
| Días ausentes | 81 días del cliente 2 en 2012-2013 | No se rellenan; quedan ausentes |
| Filas `CL` incompletas | 740 días sin `CL`: 170 del cliente 27, que nunca usa la carga controlada, y 570 de 10 clientes en bloques que terminan el 31 de marzo de 2013 | Sí: tratar al cliente 27 como hogar sin carga controlada y estimar el `CL` de los otros 10 con el año anterior |
| Consumo total | No existe como columna | Sí: calcular `GC + CL` |
| `GG` por encima de la capacidad | Algunos clientes superan la capacidad en más de un 20 % | Sí: marcar esos clientes |
| `GG` nocturno | Ruido mínimo del contador y dos clientes con días desplazados | Sí: marcar los días desplazados de los clientes 97 y 146 |
| Resolución | Media hora, frente a una hora en ERA5 | Sí: sumar a horas |

## 16. Plan de preprocesamiento de Ausgrid

Las transformaciones se programarán en `src/preprocessing/ausgrid.py`, en este orden:

| Función | Qué hace | Motivo |
|---|---|---|
| `load_ausgrid_file(path, expected_sha256, period)` | Lee el CSV omitiendo la línea de título, comprueba la huella y las columnas, elimina `Row Quality` y añade la columna `source_period`. La fecha se deja como texto | Leer los tres años de la misma forma y conservar el origen de cada fila |
| `parse_dates(df, date_format)` | Convierte la columna `date` de texto a fecha con el formato del archivo: `%d-%b-%y` para 2010-2011 y `%d/%m/%Y` para los otros dos años. Si alguna fecha no se puede interpretar o queda fuera del año del archivo, detiene el proceso | Tener las fechas de los tres años con el mismo tipo de dato |
| `validate_ausgrid(df)` | Comprueba que no haya claves cliente + fecha + categoría repetidas, celdas vacías ni negativos, y que la capacidad y el código postal sean constantes por cliente. Si algo falla, detiene el proceso | Repetir automáticamente los controles de este cuaderno |
| `estimate_missing_cl(tables)` | Recibe los tres años ya leídos. Para cada cliente con carga controlada que no tiene la fila `CL` un día en que sí tiene `GC`, crea esa fila copiando las 48 medias horas de `CL` del mismo cliente en la misma fecha del año anterior y la marca con `cl_estimated`. No estima a los clientes cuyo `CL` registrado vale siempre 0, como el 27. Si el año anterior tampoco tiene ese día, no crea la fila | Recuperar los 570 días de 2012-2013 sin subestimar la demanda |
| `find_pv_issues(tables)` | Sobre los tres años leídos, localiza los clientes con alguna media hora de `GG` por encima de 1,2 × capacidad × 0,5 h y los días con más del 20 % de su generación entre las 00:00 y las 05:00. No cambia nada; devuelve las dos listas para `flag_pv_issues` | Detectar los problemas de `GG` sobre los archivos originales, igual que en este cuaderno |
| `wide_to_long(df)` | Pasa las 48 columnas a una fila por cliente, categoría e intervalo, con la hora local del final del intervalo. La columna `0:00` se asigna a las 00:00 del día siguiente | Tener una marca de tiempo por valor |
| `build_utc_timestamps(df)` | Resuelve los días de cambio de horario y pasa la hora local a UTC con la zona `Australia/Sydney`. En el inicio del horario de verano elimina los intervalos de 02:00 a 03:00, que son ceros. En el fin, reparte a partes iguales cada valor de 02:00 a 03:00 entre las dos pasadas de esa hora. Comprueba que la energía de cada día no cambie | Poder unir con ERA5, que está en UTC |
| `pivot_categories(df)` | Pasa `GC`, `CL` y `GG` a columnas, con una fila por cliente e intervalo | Tener la demanda y la generación en la misma fila |
| `add_total_demand(df)` | Calcula `demand_kwh = GC + CL`, incluido el `CL` estimado. Si el cliente no tiene carga controlada en el año o su `CL` vale siempre 0, como los clientes 27, 37 y 281, usa `GC` y marca `has_cl = False`. Si después de estimar aún faltara `CL` en algún cliente con carga controlada, detiene el proceso | Obtener el consumo total del hogar |
| `flag_pv_issues(df, over_capacity_customers, shifted_days)` | Añade la marca `gg_unreliable` a los clientes y días que devuelve `find_pv_issues`: los 15 clientes por encima de la capacidad y los días desplazados de los clientes 97 y 146. No cambia los valores | Excluirlos al validar pvlib |
| `aggregate_to_hourly(df)` | Suma las dos medias horas de cada hora UTC, etiquetada con su final. Si a alguna hora le falta una de sus medias horas, detiene el proceso. Comprueba que la energía total no cambie | Igualar la resolución de ERA5 |
| `consolidate_periods(dfs)` | Une los tres años en una sola tabla y comprueba que la clave cliente + marca UTC sea única | Tener un único conjunto de Ausgrid |
| `prepare_ausgrid(raw_dir)` | Aplica las funciones anteriores a los tres archivos en orden | Obtener Ausgrid listo para la unión con una sola llamada |

**Unión con ERA5.** Se hace en `src/preprocessing/merge.py` con `merge_ausgrid_weather(ausgrid_hourly, weather)`, descrita en el cuaderno de ERA5. Une por la marca UTC del final de cada hora y elimina las 9 primeras horas de Ausgrid, que no tienen dato meteorológico. El conjunto final se describe en `notebooks/final_dataset.ipynb`.

**Lo que no se modifica.** Los valores de energía se conservan en kWh. No se rellenan los 81 días ausentes del cliente 2; el único valor que se estima es el `CL` de los 10 clientes de 2012-2013. No se eliminan los picos de consumo y no se corrigen los valores de `GG` marcados.

**Conjunto final.** Un único archivo en `data/processed/` con una fila por cliente y hora UTC y estas columnas: `customer`, `timestamp_utc`, `source_period`, `generator_capacity_kwp`, `postcode`, `gc_kwh`, `cl_kwh`, `gg_kwh`, `demand_kwh`, las marcas `has_cl`, `cl_estimated` y `gg_unreliable`, y las nueve variables de ERA5.